# K-center problem

The k-center problem is a facility location problem. A facility location problem is an optimization problem where the goal is to decide where to place some faciltiies (hospitals, warehouses, schools etc.) such that they serve a set of clients as efficiently as possible. The k-center problem solves the problem of choosing facilties such that the farthest served client is not too far away. For example a hospital serving two cities is better placed in the middle of those cities then in either one. Formally, the problem is defined like this: 

In the k-center problem, we are given a finite metric space $(X,d)$ where $d: P \times P \to \mathbb{R}_{\geq 0} $, and a set of points $P \subseteq X$, and are tasked to find a set of $k \in \mathbb{N}$ points(also called centers) $C \subseteq P$ such that the maximum distance from every point $p \in P$ to it's closest center is minimzed. That is, find $C$ so that $$cost(P,C)=\max_{p\in P}\min_{c \in C}d(p,c)$$ is minimized.

In [14]:
import numpy as np
from itertools import combinations

In [2]:
def manhattan_distance(x: list[int],y:list[int]):
    if len(x) != len(y):
        raise ValueError("Points has to be of same dimension")
    distance = sum(abs(i-j) for i,j in zip(x,y))
    return distance

def euclidean_distance(x: list[int], y:list[int]):
    if len(x) != len(y):
        raise ValueError("Points has to be of same dimension")
    if len(x) == 1:
        return abs(x[0]-y[0])
    distance = np.sqrt(sum(np.pow(i-j,2) for i,j in zip(x,y)))
    return distance

In [93]:
def build_dataset(n, min_val=1, max_val=200, dim=1):
    dataset = np.unique(np.random.randint(min_val, max_val, (n, dim)), axis=0)
    while len(dataset) < n:
        new_point = np.random.randint(min_val, max_val, (1, dim))
        dataset = np.unique(np.vstack([dataset, new_point]), axis=0)
    return dataset

dataset = build_dataset(n=10,dim=2)
print(f"dataset: {dataset}")


dataset: [[  8  64]
 [  9 142]
 [ 58   2]
 [ 62  23]
 [ 72 132]
 [ 97  87]
 [116 176]
 [122  31]
 [129  61]
 [142 138]]


# K-center brute force algorithm

In [34]:
k = 3
candidate_center_sets = list(combinations(dataset,k))
print(f"number of possible different clusterings with {k} centers, in a dataset of {len(dataset)} points: {len(candidate_center_sets)}")
#print(f"candidate center sets: {candidate_center_sets}")

def k_center_brute(dataset, candidate_center_sets, d, k):
    """
    Returns:
        (optimal_centers, optimal_cost): the best set of centers and its cost.
    """
    clustering_costs:dict[int,int] = {} # {clustering_index: cost}

    for i, center_set in enumerate(candidate_center_sets):

        distances_to_nearest_center = []
        
        for point in dataset:
            distance_to_nearest_center = min(
                d(point, center)
                for center in center_set
            )

            distances_to_nearest_center.append(distance_to_nearest_center)
        clustering_costs[i] = max(distances_to_nearest_center)
    
    optimal_center_set_idx = min(clustering_costs, key=lambda k: clustering_costs[k])
    optimal_centers = candidate_center_sets[optimal_center_set_idx]
    optimal_cost = clustering_costs[optimal_center_set_idx]
    return optimal_centers, optimal_cost

def extract_clustering(dataset: np.ndarray, centers, d):
    """
    Returns:
        A dict mapping each center's index to the list of points assigned to it (centers included).
    """
    clusters = {i: [] for i in range(len(centers))}
    for p in dataset:
        nearest = min(range(len(centers)), key=lambda i: d(p, centers[i])) #argmin on range(len(centers)) compared on d(p,c) / argmin pattern
        clusters[nearest].append(p)
    return clusters


optimal_centers, optimal_cost = k_center_brute(dataset=dataset, candidate_center_sets=candidate_center_sets, d=euclidean_distance, k = k)
print(f"optimal_center_set: {optimal_centers} with cost: {optimal_cost}")

clusters = extract_clustering(dataset=dataset, centers=optimal_centers, d=euclidean_distance)
print(f"clusters: {clusters}")

    


number of possible different clusterings with 3 centers, in a dataset of 10 points: 120
optimal_center_set: (array([ 73, 138]), array([135,  26]), array([158, 143])) with cost: 54.00925846556311
clusters: {0: [array([ 38, 141]), array([ 73, 138]), array([102, 147])], 1: [array([130,  72]), array([134,  80]), array([135,  26]), array([179,  21])], 2: [array([140, 157]), array([158, 143]), array([193, 145])]}


# K-center greedy 2-approximation algorithm - gonzalez

Choose the center farthest away from the rest. Given a set of centers $C$, a new center $c$ is decided by solving: $$c = \argmax_p \min_{c\in C} d(p,c)$$ $c $ is the worst served point for the current set of centers. So the intution is that if $|C| \lt k$, pick the new center to be the worst served point for the current clustering, i.e the one with the highest cost. 

In [75]:
k = 3
print(f"number of possible different clusterings with {k} centers, in a dataset of {len(dataset)} points: {len(candidate_center_sets)}")

def k_center_gonzalez(dataset: np.ndarray, d, k: int):
    if len(dataset) < k:
        raise ValueError(f"Dataset must have at least {k} points")

    first_center_idx = np.random.randint(len(dataset))
    centers = [dataset[first_center_idx]]

    for _ in range(k - 1):  # first center already chosen
        point_distances = [min(d(p, c) for c in centers) for p in dataset]
        farthest_idx = max(range(len(dataset)), key=lambda i: point_distances[i])
        centers.append(dataset[farthest_idx])

    point_distances = [min(d(p, c) for c in centers) for p in dataset]
    farthest_idx = max(range(len(dataset)), key=lambda i: point_distances[i])
    cost = point_distances[farthest_idx]
    return centers, cost

centers, cost = k_center_gonzalez(dataset=dataset,d=euclidean_distance,k=k)
print(f"greedy centers: {centers} with corresponding cost: {cost}")




number of possible different clusterings with 3 centers, in a dataset of 100 points: 120
greedy centers: [array([195,  72]), array([  4, 197]), array([14, 11])] with corresponding cost: 134.01492454200763


# Brute-force algorithm vs Gonzalez algorithm cost comparison
Expected result is that the cost of the clustering given by the Gonzales algorithm should be at most twice as much as the brute-force algorithm. The expriment is conducted as follows: 

1. Run brute-force algorithm once and save the cost
2. Run Gonzalez's algorithm x times and save the costs
3. Take the mean of the costs from 2. then the ratio between that and the cost from 1.

In [195]:
np.random.seed(1)
dataset = build_dataset(n=100, max_val = 2000, dim=2)
np.random.seed()

def k_center_gonzalez_experiment(dataset: np.ndarray, iterations: int):
    k = 3
    candidate_center_sets = list(combinations(dataset, k))
    _, optimal_cost = k_center_brute(dataset=dataset, candidate_center_sets=candidate_center_sets, d=euclidean_distance, k=k)
    #print(f"optimal cost: {optimal_cost}")
    gonzalez_costs = []
    for _ in range(iterations):
        _, cost = k_center_gonzalez(dataset=dataset,d=euclidean_distance, k=k)
        gonzalez_costs.append(cost)
        #print(f"gonzalez cost: {cost} ")
    gonzalez_costs_mean = np.mean(gonzalez_costs)
    approximation_ratio = gonzalez_costs_mean / optimal_cost
    return approximation_ratio

iterations = 10000
approximation_ratio = k_center_gonzalez_experiment(dataset=dataset, iterations = iterations) 
print(f"""Dataset size: {len(dataset)} 
Expected approximation ratio is 2. Actual ratio after {iterations} iterations: {approximation_ratio}.""")


Dataset size: 100 
Expected approximation ratio is 2. Actual ratio after 10000 iterations: 1.4296444716244023.


In [191]:
m = np.mean([1.0,2.0,3.0,4.0])
print(m)

2.5
